Generate datasets for emotion concepts

In [1]:
import json
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator, Sequence

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase
)

/Users/folusoogunlana/code/oss/emotion-concepts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct"

# Anchor every path to the project root, not the kernel's cwd — this notebook
# lives two levels down, and Jupyter front-ends disagree about what cwd is.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))  # so `import core.*` resolves

CACHE_DIR = PROJECT_ROOT / ".cache"
DATA_DIR = PROJECT_ROOT / "data"
SEED = 0

PROJECT_ROOT

PosixPath('/Users/folusoogunlana/code/oss/emotion-concepts')

In [3]:
# emotions to guide stories
EMOTIONS: tuple[str, ...] = (
    "joy", "sadness", "anger", "fear", "disgust", "surprise", "calm", "desperation", "pride", "shame", "loneliness", "excitement"
)

# topics to guide stories
TOPICS: tuple[str, ...] = (
    "a train station",
    "a job interview",
    "an old family recipe",
    "a broken bicycle",
    "the last day of school"
)

INTENSITIES: tuple[dict, ...] = (
    {
        "scenario": "paracetamol",
        "emotion": "fear",
        "steps": [
            (n, f"You took {n} paracetamol tablet this morning")
            for n in range(1, 6)
        ]
    }
    # 5 more...
)

IMPLICIT: tuple[dict, ...] = (
    {
        "text": "Your flight is delayed three hours. The wedding starts at six.",
        "emotion": "desperation"
    }
)

In [4]:
@dataclass
class Prompt:
    emotion: str
    topic: str
    index: int
    instruction: str

@dataclass
class Story:
    emotion: str
    topic: str
    index: int
    prompt: str
    text: str
    seed: int
    mentions_emotion: bool

In [5]:
from core.utils import Model
model = Model()
model.load_weights()

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 226.71it/s]


In [29]:
def build_instruction(emotion: str, topic: str) -> str:
    # print(f"emotion: {emotion}, topic: {topic}")
    if emotion == 'neutral':
        instruction = "Write a short story (max 200 words) on a topic '{topic}' where a character experiences no emotion - the story should be neutral."
    else:    
        instruction = "Write a short story (max 200 words) on a topic '{topic}' where a character experiences emotion '{emotion}'."
    return instruction
    
print(build_instruction(EMOTIONS[2], TOPICS[2]))

Write a short story (max 200 words) on a topic '{topic}' where a character experiences emotion '{emotion}'.


In [ ]:
import numpy as np

def build_prompts(emotions: list[str] = EMOTIONS, topics: list[str] = TOPICS, n_per_pair: int = 3, include_neutral: bool = True) -> list[Prompt]:
    prompts = []
    for e in list(emotions) + (["neutral"] if include_neutral else []):
        for t in topics:
            for n in range(n_per_pair):
                prompts.append(Prompt(e, t, n, instruction = build_instruction(e, t)))

    return prompts

prompts = build_prompts()

Prompt(emotion='anger', topic='a train station', index=0, instruction="Write a short story (max 200 words) on a topic '{topic}' where a character experiences emotion '{emotion}'.")
